# Ch. 2 — Preliminaries

## 2.1 Data Manipulation
- tensors = n-dim arrays, same idea as NumPy but with autograd + GPU
- `arange`, `reshape`, `zeros`/`ones`/`randn` for creation
- elementwise ops broadcast automatically when shapes are compatible

In [2]:
import torch

x = torch.arange(12, dtype=torch.float32)
x, x.shape, x.numel()

ModuleNotFoundError: No module named 'torch'

> `shape` = tuple of sizes per axis (e.g. `(3, 4)`) — tells you the *layout*.
> `numel()` = total scalar count = product of `shape` (e.g. `3*4=12`) — tells you the *storage size*.
> Same info, different granularity: use `shape` to reason about dims, `numel` to check total params/memory.

In [ ]:
X = x.reshape(3, 4)
X

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])

In [ ]:
torch.zeros((2, 3, 4)), torch.ones((2, 3, 4))

(tensor([[[0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]],
 
         [[0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]]]),
 tensor([[[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]],
 
         [[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]]]))

In [ ]:
torch.randn(3, 4)  # standard normal init

tensor([[-0.0545, -0.6747, -0.7638, -0.7530],
        [-0.1727, -1.8116,  0.6955, -0.8662],
        [ 0.2139,  0.4506,  0.8138, -1.0020]])

**Indexing/slicing** — same as Python lists, negative index = from the end.

In [ ]:
X[-1], X[1:3]

(tensor([ 8.,  9., 10., 11.]),
 tensor([[ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]))

In [ ]:
X[1, 2] = 17
X

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5., 17.,  7.],
        [ 8.,  9., 10., 11.]])

**Operations** — elementwise arithmetic + broadcasting when shapes differ.

In [ ]:
x = torch.tensor([1.0, 2, 4, 8])
y = torch.tensor([2, 2, 2, 2])
x + y, x - y, x * y, x / y, x ** y

(tensor([ 3.,  4.,  6., 10.]),
 tensor([-1.,  0.,  2.,  6.]),
 tensor([ 2.,  4.,  8., 16.]),
 tensor([0.5000, 1.0000, 2.0000, 4.0000]),
 tensor([ 1.,  4., 16., 64.]))

In [ ]:
a = torch.arange(3).reshape((3, 1))
b = torch.arange(2).reshape((1, 2))
a, b, a + b  # broadcasting: (3,1) + (1,2) -> (3,2)

(tensor([[0],
         [1],
         [2]]),
 tensor([[0, 1]]),
 tensor([[0, 1],
         [1, 2],
         [2, 3]]))

**KAN link:** a KAN layer evaluates a spline basis at every (batch, in_feature, grid_point) combo — broadcasting is exactly how that 3-way grid gets built from a `(batch, in_features)` input and a `(grid_size,)` knot vector, without writing an explicit loop.

**Saving memory** — `X[:] = ...` or `+=` avoids allocating a new tensor.

In [ ]:
Y = torch.ones(3, 4)
before = id(Y)
Y = Y + X
id(Y) == before  # False -> new tensor allocated

False

In [ ]:
Z = torch.zeros_like(X)
print('id(Z):', id(Z))
Z[:] = X + X
print('id(Z):', id(Z))  # unchanged -> in-place write

id(Z): 139980938065664
id(Z): 139980938065664


**Conversion** — `.numpy()` / `torch.from_numpy()`; `.item()` for 1-element tensors.

In [ ]:
A = X.numpy()
B = torch.from_numpy(A)
type(A), type(B)

(numpy.ndarray, torch.Tensor)

In [ ]:
a = torch.tensor([3.5])
a, a.item(), float(a), int(a)

(tensor([3.5000]), 3.5, 3.5, 3)

## 2.2 Data Preprocessing
- raw data → pandas DataFrame → handle NaNs → convert to tensor
- `fillna` / `get_dummies` for missing + categorical columns

In [ ]:
import os
import pandas as pd

os.makedirs('../data', exist_ok=True)
data_file = '../data/house_tiny.csv'
with open(data_file, 'w') as f:
    f.write('NumRooms,RoofType,Price\n')
    f.write('NA,NA,127500\n')
    f.write('2,NA,106000\n')
    f.write('4,Slate,178100\n')
    f.write('NA,NA,140000\n')

data = pd.read_csv(data_file)
data

,NumRooms,RoofType,Price
0,NaN,NaN,127500
1,2.0,NaN,106000
2,4.0,Slate,178100
3,NaN,NaN,140000


In [ ]:
inputs, targets = data.iloc[:, 0:2], data.iloc[:, 2]
inputs = pd.get_dummies(inputs, dummy_na=True)
inputs

,NumRooms,RoofType_Slate,RoofType_nan
0,NaN,False,True
1,2.0,False,True
2,4.0,True,False
3,NaN,False,True


In [ ]:
inputs = inputs.fillna(inputs.mean(numeric_only=True))
inputs

,NumRooms,RoofType_Slate,RoofType_nan
0,3.0,False,True
1,2.0,False,True
2,4.0,True,False
3,3.0,False,True


In [ ]:
X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(targets.to_numpy(dtype=float))
X, y

(tensor([[3., 0., 1.],
         [2., 0., 1.],
         [4., 1., 0.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

## 2.3 Linear Algebra
- scalar → vector → matrix → tensor (increasing dims)
- `.T` transpose, `A * B` elementwise (Hadamard), `A @ B` matmul
- `sum`/`mean` reduce; pass `axis` + `keepdims=True` to reduce along one dim only

In [ ]:
x = torch.tensor(3.0)
y = torch.tensor(2.0)
x + y, x * y, x / y, x**y

(tensor(5.), tensor(6.), tensor(1.5000), tensor(9.))

In [ ]:
x = torch.arange(3)
x, x[2], len(x), x.shape

(tensor([0, 1, 2]), tensor(2), 3, torch.Size([3]))

In [ ]:
A = torch.arange(6).reshape(3, 2)
A, A.T

(tensor([[0, 1],
         [2, 3],
         [4, 5]]),
 tensor([[0, 2, 4],
         [1, 3, 5]]))

In [ ]:
A = torch.arange(12, dtype=torch.float32).reshape(3, 4)
B = A.clone()
A, A + B, A * B  # Hadamard product

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]),
 tensor([[ 0.,  2.,  4.,  6.],
         [ 8., 10., 12., 14.],
         [16., 18., 20., 22.]]),
 tensor([[  0.,   1.,   4.,   9.],
         [ 16.,  25.,  36.,  49.],
         [ 64.,  81., 100., 121.]]))

> `A * B` (Hadamard) multiplies matching entries — same shape in, same shape out.
> `A @ B` / `torch.mm` contracts over a shared dimension — output shape changes.
**KAN link:** each KAN edge applies its own learned 1-D function to its input (elementwise-style, like Hadamard) *before* the layer sums across edges — unlike an MLP, which does the mixing via one big matmul first.

In [ ]:
A = torch.arange(6, dtype=torch.float32).reshape(2, 3)
A.sum(), A.sum(axis=0), A.sum(axis=1), A.sum(axis=1, keepdims=True)

(tensor(15.),
 tensor([3., 5., 7.]),
 tensor([ 3., 12.]),
 tensor([[ 3.],
         [12.]]))

In [ ]:
A.mean(), A.sum() / A.numel()

(tensor(2.5000), tensor(2.5000))

In [ ]:
x = torch.arange(3, dtype=torch.float32)
y = torch.ones(3, dtype=torch.float32)
x, y, torch.dot(x, y), torch.sum(x * y)  # dot product, two ways

(tensor([0., 1., 2.]), tensor([1., 1., 1.]), tensor(3.), tensor(3.))

In [ ]:
A = torch.arange(6, dtype=torch.float32).reshape(2, 3)
x = torch.arange(3, dtype=torch.float32)
A.shape, x.shape, torch.mv(A, x)  # matrix-vector product

(torch.Size([2, 3]), torch.Size([3]), tensor([ 5., 14.]))

> `torch.mv` = matrix @ vector → 1-D output. `torch.mm` = matrix @ matrix → 2-D output.
> Same underlying contraction, `mv` is just the `mm` special case with a vector.

In [ ]:
B = torch.ones(3, 4)
torch.mm(A, B)  # matrix-matrix product

tensor([[ 3.,  3.,  3.,  3.],
        [12., 12., 12., 12.]])

In [ ]:
u = torch.tensor([3.0, -4.0])
torch.norm(u), torch.abs(u).sum()  # L2 norm, L1 norm

(tensor(5.), tensor(7.))

## 2.4 Calculus
- derivative = instantaneous rate of change; gradient = vector of partials
- deep learning trains by minimizing loss via gradient descent
- (numerical illustration only — real gradients come from autograd in 2.5)

In [ ]:
def f(x):
    return 3 * x ** 2 - 4 * x

def numerical_lim(f, x, h):
    return (f(x + h) - f(x)) / h

h = 0.1
for i in range(5):
    print(f'h={h:.5f}, numerical limit={numerical_lim(f, 1, h):.5f}')
    h *= 0.1
# true derivative f'(x) = 6x - 4 -> f'(1) = 2, converges as h -> 0

h=0.10000, numerical limit=2.30000
h=0.01000, numerical limit=2.03000
h=0.00100, numerical limit=2.00300
h=0.00010, numerical limit=2.00030
h=0.00001, numerical limit=2.00003


## 2.5 Automatic Differentiation
- `requires_grad_(True)` tracks ops on a tensor for backprop
- `.backward()` computes gradients into `.grad`
- must `.zero_()` grads between calls — PyTorch accumulates by default

In [ ]:
x = torch.arange(4.0, requires_grad=True)
x.grad  # None until backward() is called

In [ ]:
y = 2 * torch.dot(x, x)  # y = 2 * sum(x_i^2)
y

tensor(28., grad_fn=<MulBackward0>)

In [ ]:
y.backward()
x.grad, x.grad == 4 * x  # dy/dx = 4x, check

(tensor([ 0.,  4.,  8., 12.]), tensor([True, True, True, True]))

In [ ]:
x.grad.zero_()  # reset before next backward
y = x.sum()
y.backward()
x.grad

tensor([1., 1., 1., 1.])

In [ ]:
x.grad.zero_()
y = x * x
y.sum().backward()  # backward on non-scalar: sum first (or pass gradient arg)
x.grad

tensor([0., 2., 4., 6.])

In [ ]:
x.grad.zero_()
y = x * x
u = y.detach()  # treat u as constant, breaks the graph
z = u * x
z.sum().backward()
x.grad == u  # gradient only flows through the detached path here

tensor([True, True, True, True])

> `.grad.zero_()` — resets accumulated gradients, graph still tracked (needed between training steps).
> `.detach()` — cuts a value out of the graph entirely; gradients won't flow through it at all.
**KAN link:** this `.backward()` mechanism is literally how a KAN's spline coefficients get trained — no different from training weights in an MLP, autograd doesn't care what the learnable function looks like.

## 2.6 Probability & Statistics
- multinomial distribution models rolling a die / sampling from a discrete dist
- empirical frequency → true probability as sample size grows (law of large numbers)

In [ ]:
from torch.distributions import multinomial

fair_probs = torch.ones(6) / 6
multinomial.Multinomial(1, fair_probs).sample()  # one die roll (one-hot count)

tensor([0., 0., 0., 1., 0., 0.])

In [ ]:
counts = multinomial.Multinomial(1000, fair_probs).sample()
counts / 1000  # relative frequency -> should approach 1/6 each

tensor([0.1870, 0.1470, 0.1780, 0.1480, 0.1750, 0.1650])

---
**Takeaway:** tensors + broadcasting cover the data plumbing; autograd (2.5) is the piece that actually matters going forward — everything in later chapters (loss, backprop, optimizers) builds on `.backward()`.